# 데이터 집계 — groupby · agg · pivot_table


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 분할–적용–결합을 이해하고 주문 건수·판매 수량·고유 고객 수를 구분하며, 목적에 맞는 집계표를 만듭니다.

## 처음 배우는 분을 위한 안내

groupby는 같은 범주별로 행을 묶습니다. sum은 합계, mean은 평균, size는 행 수, count는 비어 있지 않은 값 수, nunique는 서로 다른 값 수입니다. agg는 한 표에 여러 집계값을 모읍니다. 피벗은 두 범주를 표의 행·열로 펼치는 방법입니다.

**학습 순서:** 문제 1~5는 필수 기초, 결합·피벗·검증을 함께 쓰는 6~12는 선택 도전입니다. df는 한 행이 주문 한 건이며 revenue는 주문 금액, quantity는 상품 수량입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요. 예제는 순서대로 이어집니다. 독립 실행이 필요한 경우 이 준비 코드를 먼저 실행하세요.

```python
import pandas as pd

# 학습용 가상 주문 데이터. 실제 고객 데이터가 아닙니다.
df = pd.DataFrame({
    'order_id': list(range(101, 113)),
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-01-06', '2024-01-07', '2024-02-01',
                   '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05'],
    'customer_id': [1, 2, 1, 3, 4, 2, 5, 1, 3, 4, 5, 2],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Tech', 'Book', 'Food', 'Tech', 'Book'],
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 1, 3, 2, 1, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 1000, 200, 100, 1000, 200],
    'discount': [0, 0.1, 0, 0.2, 0, 0.1, 0, 0, 0.1, 0, 0.2, 0],
    'payment_method': ['Card', 'Wallet', 'Card', 'COD', 'Card', 'Wallet',
                       'COD', 'Card', 'Wallet', 'Card', 'COD', 'Card'],
    'delivery_days': [1, 4, 3, 7, 2, 5, 8, 2, 6, 3, 9, 4],
    'customer_rating': [4.5, 4.0, 4.8, 2.5, 4.2, 3.5, 2.0, 4.6, 3.0, 4.4, 2.2, 4.1],
    'revenue': [200, 180, 300, 800, 400, 1800, 100, 1000, 540, 200, 800, 400]
})
df['order_date'] = pd.to_datetime(df['order_date'])
df_original = df.copy(deep=True)

customer_master = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 6],
    'membership_tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Bronze'],
    'region': ['South', 'North', 'East', 'North', 'West']
})
customer_summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'total_orders': [3, 3, 2, 2, 2],
    'total_spent': [1500, 2380, 1340, 600, 900],
    'avg_order_value': [500, 2380 / 3, 670, 300, 450]
})
```

## groupby를 이해하는 질문

'지역별 매출 합계는?'에서 그룹 기준은 region, 집계 대상은 revenue, 연산은 sum입니다. Split은 같은 지역끼리 묶기, Apply는 각 그룹의 합계 계산, Combine은 결과를 한 표로 모으기입니다. groupby()만 호출하면 GroupBy 객체이며 실제 집계 결과가 아닙니다.

```python
region_sales = df.groupby('region', as_index=False)['revenue'].sum()
print(region_sales)
# East 1100, North 2380, South 3240 (표시 순서는 정렬 설정에 따라 다름)
```

## 주문 수·수량·고객 수

| 연산 | 의미 | 주의점 |
| --- | --- | --- |
| size() | 그룹의 행 수 | 한 행이 주문 한 건일 때 주문 건수 |
| count() | 선택 열의 비결측 개수 | ID가 누락되면 행 수보다 작음 |
| quantity.sum() | 판매 수량 | 주문 건수와 다름 |
| customer_id.nunique() | 고유 고객 수 | 반복 구매 고객은 한 번만 계산 |

평균은 결측을 제외한 값으로 계산됩니다. 합계에 결측이 있어도 계산되며 전부 결측인 그룹을 0으로 볼지 주의가 필요합니다. 필요하면 sum(min_count=1)로 전부 결측인 경우 결측으로 유지할 수 있습니다.

## agg와 이름 있는 집계

동일 열에 여러 함수를 적용하거나 열마다 다른 함수를 쓸 수 있습니다. 이름 있는 집계는 결과 열이 무엇을 뜻하는지 명확하게 만듭니다. as_index=False와 reset_index()는 그룹 키를 일반 열로 만드는 대표 방법이며 상황에 맞게 하나를 선택합니다.

```python
report = df.groupby('product_category', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_quantity=('quantity', 'sum'),
    order_count=('order_id', 'size'),
    unique_customers=('customer_id', 'nunique')
)
print(report)
```

## 여러 기준과 누락된 그룹

groupby(['region', 'product_category'])는 지역·카테고리 조합별로 집계합니다. 기본 dropna=True는 그룹 키가 결측인 행을 제외하므로 전체 합계와 집계 합계가 다를 수 있습니다. 필요하면 dropna=False를 명시하세요. 범주형 키의 미관측 범주까지 표시할지는 observed를 명시해 의도를 고정합니다.

```python
two_keys = df.groupby(['region', 'product_category'],
                      dropna=False, observed=True)['revenue'].sum()
print(two_keys.reset_index())
```

## 고객 등급 집계의 분석 단위

고객 요약표와 고객 마스터를 결합하면 한 행이 고객입니다. 이때 total_spent의 평균은 고객당 평균 구매금액이며 주문별 평균 매출과 다릅니다. total_spent의 합을 total_orders의 합으로 나누면 가중된 주문당 평균을 구할 수 있습니다. 고객별 avg_order_value를 단순 평균한 값과 구분하세요.

```python
customer_view = customer_summary.merge(
    customer_master[['customer_id', 'membership_tier']],
    on='customer_id', how='left', validate='one_to_one')
tier_report = customer_view.groupby('membership_tier', dropna=False).agg(
    total_sales=('total_spent', 'sum'),
    avg_spent=('total_spent', 'mean'),
    customer_count=('customer_id', 'nunique')
)
print(tier_report)
```

## pivot_table: 집계를 표로 펼치기

index는 행의 그룹, columns는 열의 그룹, values는 집계 값, aggfunc는 집계 방법입니다. aggfunc를 생략하면 평균이 기본이므로 매출 합계를 원할 때 sum을 명시합니다. fill_value는 집계 결과의 빈 칸을 채우며 원본의 누락값을 자동으로 해결하지 않습니다. 관측이 없는 조합을 0으로 볼 수 있을 때만 사용하세요.

```python
pivot = pd.pivot_table(df, index='region', columns='product_category',
                       values='revenue', aggfunc='sum', fill_value=0,
                       observed=True)
print(pivot)
print(pivot.to_numpy().sum())  # 6720
equivalent = df.groupby(['region', 'product_category'])['revenue'].sum().unstack(fill_value=0)
print(equivalent)
```

## 합계·평균과 검증

margins=True는 전체 집계를 추가합니다. 평균 피벗의 All은 원본 행에 대한 전체 평균이며 보이는 그룹 평균들의 단순 평균이 아닙니다. pivot은 키 조합이 중복되면 실패하지만 pivot_table은 지정한 함수로 집계할 수 있습니다.

검증은 전체 합계 보존, 그룹별 표본 수, 누락된 키, 중복 결합을 함께 확인합니다. 표를 그럴듯하게 만드는 것보다 분모가 맞는지 확인하는 것이 먼저입니다.

```python
with_total = pd.pivot_table(df, index='region', columns='payment_method',
    values='revenue', aggfunc='sum', fill_value=0, margins=True)
print(with_total.loc['All', 'All'])  # 6720
assert df['revenue'].sum() == report['total_revenue'].sum()
```

## 복습 순서

1. 개념과 입력·출력의 분석 단위를 확인합니다.
2. 작은 예제에서 결과를 손으로 예상합니다.
3. 연습용 노트북의 준비 셀을 실행하고 문제를 풉니다.
4. 결과 수치와 점검 기준을 비교하고 이유를 기록합니다.
5. 기본 문제를 마친 뒤 선택 도전을 진행합니다. 입력을 바꾸면 결과도 달라지는 이유를 설명합니다.

## 선택 도전: 한 주문 금액만 바꿔 집계 검산하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
extra_df = df.copy(deep=True)
extra_df.loc[0, 'revenue'] = 250
```

### 진행 순서

1. extra_df를 product_category별로 묶어 revenue 합계를 구합니다.
2. 세 카테고리 합계를 다시 더해 extra_df 전체 합계와 비교합니다.
3. 원래 df의 합계가 그대로인지 확인합니다.

**완료 기준:** Food 850, Book 1520, Tech 4400, 전체 6770. df 전체 합계는 6720으로 유지.

### 선택 도전 풀이

```python
extra_totals = extra_df.groupby('product_category')['revenue'].sum()
print(extra_totals)
print(extra_totals.sum(),df['revenue'].sum())
assert extra_totals['Food']==850 and extra_totals.sum()==6770
assert df['revenue'].sum()==6720
```